# Scratch notebook for the Managed Airflow Service lab

This notebook deletes any data and Iceberg tables created as part of testing the medallion architecture PySpark scripts so that you can validate the orchestration of creation of the bronze, silver, gold and platinum layers by Airflow as part of the DAG execution.

## Section 1: Clean up lakehouse and catalogs before running the Airflow DAG

In [ ]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
LOCATION = "us-central1"
STAGE_BUCKET_NAME = f"froyo-lakehouse-staging-{PROJECT_NBR}"
ICEBERG_LAKEHOUSE_BUCKET_NAME = f"froyo_iceberg_lakehouse_catalog_{PROJECT_NBR}"
ICEBERG_CATALOG_NAME="froyo_iceberg_catalog"
ICEBERG_NAMESPACE="froyo_ns"
APP_NAME="froyo_app"

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session
from pyspark.sql import functions as F

REST_API_VERSION="v1beta"

# Create the Dataproc Serverless session.
s8s_spark_session = Session()

# Serverless runtime at authoring was 3.0 with Iceberg 1.10
s8s_spark_session.runtime_config.properties[f"spark.sql.defaultCatalog"] = ICEBERG_CATALOG_NAME
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}"] = "org.apache.iceberg.spark.SparkCatalog"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.type"] = "rest"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.uri"] = f"https://biglake.googleapis.com/iceberg/{REST_API_VERSION}/restcatalog"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.warehouse"] = f"gs://{ICEBERG_LAKEHOUSE_BUCKET_NAME}"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.io-impl"] = "org.apache.iceberg.gcp.gcs.GCSFileIO"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.header.x-goog-user-project"] = PROJECT_ID
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.rest.auth.type"] = "org.apache.iceberg.gcp.auth.GoogleAuthManager"
s8s_spark_session.runtime_config.properties[f"spark.sql.extensions"] = "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
s8s_spark_session.runtime_config.properties[f"spark.sql.catalog.{ICEBERG_CATALOG_NAME}.rest-metrics-reporting-enabled"] = "false"
s8s_spark_session.runtime_config.properties["spark.dataproc.lineage.enabled"] = "true"
s8s_spark_session.runtime_config.properties["spark.openlineage.transport.type"] = "gcplineage"
s8s_spark_session.runtime_config.properties["spark.extraListeners"] = "io.openlineage.spark.agent.OpenLineageSparkListener"
s8s_spark_session.runtime_config.properties["spark.sql.repl.eagerEval.enabled"] = "True" # Property values should be strings
s8s_spark_session.runtime_config.properties["spark.openlineage.namespace"] = "froyo_spark_jobs"
s8s_spark_session.runtime_config.properties["spark.log.level.io.openlineage"] = "DEBUG"



spark = (DataprocSparkSession.builder
    .appName(APP_NAME)
    .dataprocSessionConfig(s8s_spark_session)
    .getOrCreate())

In [ ]:
spark.sql("SHOW TABLES IN froyo_ns").show(truncate=False)

In [ ]:
# Drop any existing tables in case of a rerun

# 1. Get the dataframe containing the tables
tables_df = spark.sql("SHOW TABLES IN froyo_ns")

# 2. Collect rows to the driver
# (SHOW TABLES outputs columns: 'namespace', 'tableName', and 'isTemporary')
tables_list = tables_df.collect()

print(f"Found {len(tables_list)} targets in 'froyo_ns'. Starting iterative drop...")

# 3. Loop and drop
for row in tables_list:
    table_name = row['tableName']
    is_temp = row['isTemporary']

    # Fully qualify the name to ensure you drop from the correct namespace
    fq_name = f"froyo_ns.{table_name}"

    try:
        if is_temp:
            # If it's a temporary view, use DROP VIEW
            print(f"Dropping temporary view: {table_name}")
            spark.sql(f"DROP TEMPORARY VIEW IF EXISTS {table_name}")
        else:
            # Standard or Iceberg table
            print(f"Dropping table: {fq_name}")
            spark.sql(f"DROP TABLE IF EXISTS {fq_name}")

    except Exception as e:
        print(f"⚠️ Failed to drop {table_name}: {e}")

print("Iterative drop process complete.")

In [ ]:
!gsutil -m rm -rf gs://$ICEBERG_LAKEHOUSE_BUCKET_NAME/froyo-raw

In [ ]:
!gsutil -m rm -rf gs://$ICEBERG_LAKEHOUSE_BUCKET_NAME/froyo-recipe-pdfs

In [ ]:
!gsutil -m rm -rf gs://$ICEBERG_LAKEHOUSE_BUCKET_NAME/froyo_ns

## Section 2: [Optional] Run a few catalog and data queries via Spark after running the Airflow DAG

In [ ]:
spark.sql("SHOW TABLES IN froyo_ns").show(truncate=False)

In [ ]:
spark.sql("SELECT * from froyo_ns.g_orders_enriched limit 2").show(truncate=False)

In [ ]:
%%bigquery

SELECT * FROM `lakehouse-solutions-build.froyo_iceberg_lakehouse_catalog_30466744069.froyo_ns.p_rdm_customer_segmentation_by_age` LIMIT 2